In [12]:
import pandas as pd
df = pd.read_csv("archive/creditcard.csv")
df.head

<bound method NDFrame.head of             Time         V1         V2        V3        V4        V5  \
0            0.0  -1.359807  -0.072781  2.536347  1.378155 -0.338321   
1            0.0   1.191857   0.266151  0.166480  0.448154  0.060018   
2            1.0  -1.358354  -1.340163  1.773209  0.379780 -0.503198   
3            1.0  -0.966272  -0.185226  1.792993 -0.863291 -0.010309   
4            2.0  -1.158233   0.877737  1.548718  0.403034 -0.407193   
...          ...        ...        ...       ...       ...       ...   
284802  172786.0 -11.881118  10.071785 -9.834783 -2.066656 -5.364473   
284803  172787.0  -0.732789  -0.055080  2.035030 -0.738589  0.868229   
284804  172788.0   1.919565  -0.301254 -3.249640 -0.557828  2.630515   
284805  172788.0  -0.240440   0.530483  0.702510  0.689799 -0.377961   
284806  172792.0  -0.533413  -0.189733  0.703337 -0.506271 -0.012546   

              V6        V7        V8        V9  ...       V21       V22  \
0       0.462388  0.239599  0.

In [13]:
print(df.shape)
print(df['Class'].value_counts())

(284807, 31)
Class
0    284315
1       492
Name: count, dtype: int64


Z-Score 


In [14]:
import numpy as np
df['Amount_zscore'] = (df['Amount'] - df['Amount'].mean()) / df['Amount'].std()
df['zscore_flag'] = (df['Amount_zscore'].abs() > 3).astype(int)
print(df['zscore_flag'].value_counts())

zscore_flag
0    280731
1      4076
Name: count, dtype: int64


In [15]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(df['Class'], df['zscore_flag']))
print(classification_report(df['Class'], df['zscore_flag']))

[[280250   4065]
 [   481     11]]
              precision    recall  f1-score   support

           0       1.00      0.99      0.99    284315
           1       0.00      0.02      0.00       492

    accuracy                           0.98    284807
   macro avg       0.50      0.50      0.50    284807
weighted avg       1.00      0.98      0.99    284807



Isolation Forest

In [16]:
from sklearn.ensemble import IsolationForest

features = df.drop(columns=['Class', 'Amount_zscore', 'zscore_flag'])

iso = IsolationForest(n_estimators=100, contamination=0.0017, random_state=42)
iso.fit(features)

df['iso_pred'] = iso.predict(features)
df['iso_flag'] = (df['iso_pred'] == -1).astype(int)

print(confusion_matrix(df['Class'], df['iso_flag']))
print(classification_report(df['Class'], df['iso_flag']))

[[283955    360]
 [   367    125]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    284315
           1       0.26      0.25      0.26       492

    accuracy                           1.00    284807
   macro avg       0.63      0.63      0.63    284807
weighted avg       1.00      1.00      1.00    284807



In [17]:
from sklearn.preprocessing import StandardScaler

features_scaled = features.copy()
scaler = StandardScaler()
features_scaled[['Time', 'Amount']] = scaler.fit_transform(features[['Time', 'Amount']])

iso3 = IsolationForest(n_estimators=100, contamination=0.0017, random_state=42)
iso3.fit(features_scaled)
df['iso_flag3'] = (iso3.predict(features_scaled) == -1).astype(int)
print(classification_report(df['Class'], df['iso_flag3']))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    284315
           1       0.26      0.25      0.26       492

    accuracy                           1.00    284807
   macro avg       0.63      0.63      0.63    284807
weighted avg       1.00      1.00      1.00    284807



In [18]:
for c in [0.001, 0.003, 0.005, 0.01]:
    iso_test = IsolationForest(n_estimators=100, contamination=c, random_state=42)
    iso_test.fit(features)
    flag = (iso_test.predict(features) == -1).astype(int)
    print(f"--- contamination={c} ---")
    print(classification_report(df['Class'], flag))

--- contamination=0.001 ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    284315
           1       0.29      0.17      0.21       492

    accuracy                           1.00    284807
   macro avg       0.64      0.58      0.61    284807
weighted avg       1.00      1.00      1.00    284807

--- contamination=0.003 ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    284315
           1       0.21      0.36      0.26       492

    accuracy                           1.00    284807
   macro avg       0.60      0.68      0.63    284807
weighted avg       1.00      1.00      1.00    284807

--- contamination=0.005 ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    284315
           1       0.15      0.44      0.23       492

    accuracy                           0.99    284807
   macro avg       0.58      0.72      0.61 

In [19]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report

# Train final model
behavioral_model = IsolationForest(n_estimators=100, contamination=0.003, random_state=42)
behavioral_model.fit(features)

def score_transaction(transaction_row, model=behavioral_model):
    """
    Takes a single transaction (as a DataFrame row with same columns as `features`)
    Returns a dict: anomaly flag, raw score, and human-readable verdict
    """
    pred = model.predict(transaction_row)[0]
    raw_score = model.decision_function(transaction_row)[0]  # lower = more anomalous
    
    is_anomaly = (pred == -1)
    
    return {
        "is_anomaly": bool(is_anomaly),
        "anomaly_score": round(float(raw_score), 4),
        "verdict": "FLAGGED - unusual transaction pattern" if is_anomaly else "NORMAL"
    }

# Test it on a single row
sample = features.iloc[[0]]
print(score_transaction(sample))

{'is_anomaly': False, 'anomaly_score': 0.2486, 'verdict': 'NORMAL'}


KYC Agent

In [20]:
%pip install rapidfuzz

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [21]:
from rapidfuzz import fuzz

# Mock watchlist (tokenized entities flagged for sanctions/fraud history)
watchlist = ["PERSON_0042", "PERSON_0198", "PERSON_0753", "PERSON_1200"]

def kyc_check(entity_token, watchlist=watchlist, threshold=90):
    """
    Checks a tokenized entity against the watchlist using fuzzy matching.
    Returns match status, closest match, and similarity score.
    """
    best_match = None
    best_score = 0
    
    for entry in watchlist:
        score = fuzz.ratio(entity_token, entry)
        if score > best_score:
            best_score = score
            best_match = entry
    
    is_match = best_score >= threshold
    
    return {
        "match_found": is_match,
        "matched_entity": best_match if is_match else None,
        "similarity_score": best_score,
        "verdict": f"WATCHLIST MATCH - {best_match}" if is_match else "NO MATCH"
    }

# Test
print(kyc_check("PERSON_0042"))   # exact match
print(kyc_check("PERSON_0043"))   #close match

{'match_found': True, 'matched_entity': 'PERSON_0042', 'similarity_score': 100.0, 'verdict': 'WATCHLIST MATCH - PERSON_0042'}
{'match_found': True, 'matched_entity': 'PERSON_0042', 'similarity_score': 90.9090909090909, 'verdict': 'WATCHLIST MATCH - PERSON_0042'}


Testing Transaction

In [24]:
def investigate_transaction(entity_token, transaction_row):
    """
    Single entry point for the Aggregator.
    Runs both KYC/Sanctions and Behavioral checks on one case.
    """
    kyc_result = kyc_check(entity_token)
    behavioral_result = score_transaction(transaction_row)
    
    return {
        "entity_token": entity_token,
        "kyc_agent": kyc_result,
        "behavioral_agent": behavioral_result
    }

In [25]:

test_index = df[df['Class'] == 1].index[0]
test_transaction = features.loc[[test_index]]

test_entity = "PERSON_0042"  # deliberately using a watchlist entity to test both flags firing together

result = investigate_transaction(test_entity, test_transaction)
print(result)

{'entity_token': 'PERSON_0042', 'kyc_agent': {'match_found': True, 'matched_entity': 'PERSON_0042', 'similarity_score': 100.0, 'verdict': 'WATCHLIST MATCH - PERSON_0042'}, 'behavioral_agent': {'is_anomaly': False, 'anomaly_score': 0.0968, 'verdict': 'NORMAL'}}
